In [1]:
# base libraries
import numpy as np
import pandas as pd

# plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# regular expression module in python to find all sequences of digits in a given string
import re

# data and preprocessing libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import preprocessing

import os
from datetime import datetime, timedelta
import math
from scipy import stats
from scipy.stats import ttest_ind
from scipy.interpolate import griddata
from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import RBFInterpolator
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler

# Error evaluation libraries
from sklearn.metrics import confusion_matrix

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Error evaluation libraries
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import mean_absolute_percentage_error as mape
from sklearn.metrics import r2_score
from sklearn.metrics import classification_report

import researchpy as rp

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.utils.validation")

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from tabulate import tabulate

In [2]:
# Specify the directory and filename
save_dir = "./Images/"

In [3]:
study_data = pd.read_csv('ISU_data_by_0.1_sec.csv')

In [4]:
study_data.shape

(67660594, 29)

### Data Analysis

In [5]:
# Display the head of the dataset
study_data.head()

,User_X,User_Y,User_Z,User_Pitch,User_Yaw,User_Roll,U_X,U_Y,U_Z,GazeOrigin_X,...,AGV_Z,AGV_Pitch,AGV_Yaw,AGV_Roll,AGV_spd,Timestamp,quantile,AGVname,PID,DRate
0,5051.235,8610.83,-209.725,-21.416662,-92.196247,-0.159610,60.524000,-107.387000,156.022000,4990.711200,...,-294.143667,0.118615,89.999444,0.002393,0.000238,15:02:48,10,AGV2,1,High
1,5051.235,8610.83,-209.725,-21.168361,-92.384752,0.290698,60.493000,-108.133857,156.034571,4990.742143,...,-285.730143,0.188209,89.999339,-0.000460,0.154715,15:02:49,1,AGV2,1,High
2,5051.235,8610.83,-209.725,-21.240655,-92.761660,0.339485,60.271857,-108.963714,156.053429,4990.963429,...,-278.508286,0.319149,89.999565,-0.001784,0.526054,15:02:49,2,AGV2,1,High
3,5051.235,8610.83,-209.725,-21.004733,-93.120707,0.251285,59.831714,-109.665143,156.194286,4991.403286,...,-278.478571,0.467534,90.000198,-0.001954,0.893006,15:02:49,3,AGV2,1,High
4,5051.235,8610.83,-209.725,-20.717933,-91.225685,0.512156,59.295143,-110.065286,156.430571,4991.940286,...,-282.613429,0.620704,90.001157,-0.001464,1.263035,15:02:49,4,AGV2,1,High


In [6]:
# Get the list of columns in the dataset
columns_list = study_data.columns.tolist()

# Print the list of columns
print(columns_list)

['User_X', 'User_Y', 'User_Z', 'User_Pitch', 'User_Yaw', 'User_Roll', 'U_X', 'U_Y', 'U_Z', 'GazeOrigin_X', 'GazeOrigin_Y', 'GazeOrigin_Z', 'GazeDirection_X', 'GazeDirection_Y', 'GazeDirection_Z', 'Confidence', 'Gaze_on_AGV', 'AGV_X', 'AGV_Y', 'AGV_Z', 'AGV_Pitch', 'AGV_Yaw', 'AGV_Roll', 'AGV_spd', 'Timestamp', 'quantile', 'AGVname', 'PID', 'DRate']


In [7]:
# Check for NaN values in the DataFrame
nan_counts = study_data.isna().sum()

# Display columns with NaN counts greater than 0
nan_columns = nan_counts[nan_counts > 0]
print("Columns with NaN values:")
print(nan_columns)

Columns with NaN values:
Series([], dtype: int64)


In [8]:
# Removing rows with missing values
data_cleaned = study_data.dropna()

# Checking the shape of the cleaned dataset to confirm the rows were removed
data_cleaned.shape

study_data = data_cleaned

In [ ]:
model_data = pd.read_csv('Per_Interaction_Data_Merged.csv')

model_data = model_data.drop(columns=['AGV_Approaching.1','User_Trajectory.1','Gaze_on_AGV.1','Cross_First.1','User_Relative_Speed.1','Frechet_Distance.1'])

model_data.head()

In [ ]:
# AGVname column in this dataset has AGV[x], and we want to get rid of AGV to be consistent with rest of the datasets
def extract_number(string):
    # Find all sequences of digits in the string
    numbers = re.findall(r'\d+', string)
    # Join the found numbers (if any) an convert to integer
    return int(numbers[0]) if numbers else None 

study_data['AGVname'] = study_data['AGVname'].apply(extract_number)

### Adding Trust as a Ground Truth Value

In [ ]:
study_data['Trust'] = ""

In [ ]:
# Create a mapping dictionary from model_data
trust_dict = model_data.set_index(['AGVname', 'DRate', 'PID'])['Trust'].to_dict()

# Map the Trust values from model_data to data_by_sec based on the unique pairs ('AGVname', 'SCN', 'PID')
study_data['Trust'] = study_data.set_index(['AGVname', 'DRate', 'PID']).index.map(trust_dict)

In [ ]:
study_data.head()

### Adding Expectancy as a Ground Truth Value

In [ ]:
study_data['Expect'] = ""

In [ ]:
# Create a mapping dictionary from model_data
expect_dict = model_data.set_index(['AGVname', 'DRate', 'PID'])['Expect'].to_dict()

# Map the Trust values from model_data to data_by_sec based on the unique pairs ('AGVname', 'SCN', 'PID')
study_data['Expect'] = study_data.set_index(['AGVname', 'DRate', 'PID']).index.map(expect_dict)

In [ ]:
study_data.head()

### Adding Safety as a Ground Truth Value

In [ ]:
study_data['Safe'] = ""

In [ ]:
# Create a mapping dictionary from model_data
safety_dict = model_data.set_index(['AGVname', 'DRate', 'PID'])['Safe'].to_dict()

# Map the Trust values from model_data to data_by_sec based on the unique pairs ('AGVname', 'SCN', 'PID')
study_data['Safe'] = study_data.set_index(['AGVname', 'DRate', 'PID']).index.map(safety_dict)

In [ ]:
study_data.head()

### Adding Comfort as a Ground Truth Value

In [ ]:
study_data['Comfort'] = ""

In [ ]:
# Create a mapping dictionary from model_data
comfort_dict = model_data.set_index(['AGVname', 'DRate', 'PID'])['Comfort'].to_dict()

# Map the Trust values from model_data to data_by_sec based on the unique pairs ('AGVname', 'SCN', 'PID')
study_data['Comfort'] = study_data.set_index(['AGVname', 'DRate', 'PID']).index.map(comfort_dict)

In [ ]:
study_data.head()

### Adding AGV_Approaching Direction as a Column

In [ ]:
study_data['AGV_Approaching'] = ""

In [ ]:
# Create a mapping dictionary from model_data
AGV_Approaching_dict = model_data.set_index(['AGVname', 'DRate', 'PID'])['AGV_Approaching'].to_dict()

# Map the Trust values from model_data to data_by_sec based on the unique pairs ('AGVname', 'SCN', 'PID')
study_data['AGV_Approaching'] = study_data.set_index(['AGVname', 'DRate', 'PID']).index.map(AGV_Approaching_dict)

In [ ]:
study_data.head()

### Adding User_Trajectory as a Column

In [ ]:
study_data['User_Trajectory'] = ""

In [ ]:
# Create a mapping dictionary from model_data
User_Trajectory_dict = model_data.set_index(['AGVname', 'DRate', 'PID'])['User_Trajectory'].to_dict()

# Map the Trust values from model_data to data_by_sec based on the unique pairs ('AGVname', 'SCN', 'PID')
study_data['User_Trajectory'] = study_data.set_index(['AGVname', 'DRate', 'PID']).index.map(User_Trajectory_dict)

In [ ]:
study_data.head()

### Dropping Timestamp Column
#### When working with machine learning algorithms, we mostly need numerical data; that's why we'd like to change the format of the timestamp to the actual second of the day the particular interaction happened.

In [ ]:
# Convert the 'Timestamp' column to pandas datetime object
'''
study_data['Timestamp'] = pd.to_datetime(study_data['Timestamp'])

# Convert the 'Timestamp' column to seconds
study_data['Time_Seconds'] = study_data['Timestamp'].dt.hour * 3600 + study_data['Timestamp'].dt.minute * 60 + study_data['Timestamp'].dt.second

study_data = study_data.drop(columns=['Timestamp'])

study_data.head()
'''

### Swap the values 1 and 2 in the 'AGVname' column

In [10]:
# Swap the values 1 and 2 in the 'AGVname' column
study_data.AGVname.replace([2, 1], ['first', 'second'], inplace=True)
study_data.AGVname.replace(['first', 'second'],[1, 2], inplace=True)

### Adding 'User_Relative_Speed' as a column

In [ ]:
# We'd add three new columns to the ISU_data_by_0.1_sec
study_data = study_data.assign(User_X_Difference = None, User_Y_Difference = None, User_Relative_Speed = None)

study_data['User_X_Difference'] = study_data.groupby(['PID', 'AGVname', 'DRate'])['User_X'].diff().fillna(0).abs()
study_data['User_Y_Difference'] = study_data.groupby(['PID', 'AGVname', 'DRate'])['User_Y'].diff().fillna(0).abs()

study_data['User_Relative_Speed'] = np.sqrt(study_data['User_X_Difference']**2 + study_data['User_Y_Difference']**2)/0.1

In [ ]:
study_data.iloc[1230:1240]

### Adding AGV in FOV as a column

In [ ]:
# Subtracting User_X from AGV_X
study_data['AGV_User_X_Difference'] = study_data.apply(lambda x: x['AGV_X'] - x['User_X'], axis = 1)

# Subtracting User_Y from AGV_Y
study_data['AGV_User_Y_Difference'] = study_data.apply(lambda x: x['AGV_Y'] - x['User_Y'], axis = 1)

# Calculating the arctan of AGV_User_X_Difference/AGV_User_Y_Difference
study_data['arctan_of_difference'] = np.degrees(np.arctan(study_data['AGV_User_X_Difference']/study_data['AGV_User_Y_Difference']))
study_data['arctan_of_difference'] = np.where(study_data['AGV_User_Y_Difference'] < 0, study_data['arctan_of_difference'] + 180, study_data['arctan_of_difference'])

# If |arctan_of_difference - User_Yaw| > 45, then AGV_in_Fov No, otherwise Yes
study_data['AGV_in_Fov'] = np.where(abs(study_data['arctan_of_difference']-study_data['User_Yaw']) > 45, int(0), int(1))

study_data.loc[:,['AGV_X', 'AGV_Y', 'User_X', 'User_Y','AGV_User_X_Difference', 'AGV_User_Y_Difference', 'arctan_of_difference', 'User_Yaw', 'AGV_in_Fov','Gaze_on_AGV']].head()

#### Fact-checking FOV's calculation accuracy

In [ ]:
print(study_data.groupby('AGV_in_Fov')['Gaze_on_AGV'].value_counts())

In [ ]:
np.degrees(np.arctan(-2))

### Adding Fretchet Distance as a column

In [ ]:
import sys
from frechetdist import frdist
from concurrent.futures import ThreadPoolExecutor
from typing import List, Tuple

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import sys
from typing import List, Tuple

# Increasing the recursion limit
sys.setrecursionlimit(10000)


# Asking for the number of coordinate points in each horizon
n_points = int(input("Enter the number of coordinate points to be used in calculating the frechet distance: "))

# function to calculate Euclidean Distance
def euclidean_distance(p1, p2):
    return np.linalg.norm(p1 - p2, axis=1)

# Function to generate expected trajectory points from the current position to the end
def generate_expected_trajectory(start: Tuple[float, float], end: Tuple[float, float], num_points: int) -> np.ndarray:
    return np.column_stack((
        np.linspace(start[0], end[0], num_points),
        np.linspace(start[1], end[1], num_points)
    ))

# Function to select n points from the actual path
def select_actual_path_points(current_position: np.ndarray, path: np.ndarray, num_points: int) -> np.ndarray:
    distances = euclidean_distance(path, current_position)
    current_index = np.argmin(distances)
    end_index = min(current_index + num_points, len(path))  # Ensuring we don't go beyond the list's length
    return path[current_index:end_index]

# Fréchet Distance calculation
def frechet_distance(P: np.ndarray, Q: np.ndarray) -> float:
    ca = np.full((len(P), len(Q)), -1.0)
    return _c(ca, P, Q, len(P)-1, len(Q)-1)

def _c(ca: np.ndarray, P: np.ndarray, Q: np.ndarray, i: int, j: int) -> float:
    if ca[i, j] > -1:
        return ca[i, j]
    elif i == 0 and j == 0:
        ca[i, j] = euclidean_distance(P[0:1], Q[0:1])[0]
    elif i > 0 and j == 0:
        ca[i, j] = max(_c(ca, P, Q, i-1, 0), euclidean_distance(P[i:i+1], Q[0:1])[0])
    elif i == 0 and j > 0:
        ca[i, j] = max(_c(ca, P, Q, 0, j-1), euclidean_distance(P[0:1], Q[j:j+1])[0])
    elif i > 0 and j > 0:
        ca[i, j] = max(min(_c(ca, P, Q, i-1, j), _c(ca, P, Q, i-1, j-1), _c(ca, P, Q, i, j-1)), euclidean_distance(P[i:i+1], Q[j:j+1])[0])
    else:
        ca[i, j] = float('inf')
    return ca[i, j]

# Processing each interaction group
def process_interaction_group(group: pd.DataFrame) -> dict:
    actual_path = group[['User_X', 'User_Y']].values
    end_point = actual_path[-1]  # Endpoint to the last coordinate of the group
    frechet_distances = []

    for current_position in actual_path:
        expected_trajectory = generate_expected_trajectory(current_position, end_point, n_points)
        selected_actual_path = select_actual_path_points(current_position, actual_path, n_points)
        frechet_dist = frechet_distance(expected_trajectory, selected_actual_path)
        frechet_distances.append(frechet_dist)

    return {
        'PID': group['PID'].iloc[0],
        'AGVname': group['AGVname'].iloc[0],
        'DRate': group['DRate'].iloc[0],
        'Point_ID': group['Point_ID'].values,
        'Frechet_Distance': frechet_distances
    }

# Grouping data
grouped = study_data.groupby(['PID', 'AGVname', 'DRate'])

# Adding a sequential identifier within each group in the original dataset
study_data['Point_ID'] = study_data.groupby(['PID', 'AGVname', 'DRate']).cumcount()

# Creating a function to handle the parallel processing
def parallel_process_group(group_tuple):
    key, group = group_tuple
    return process_interaction_group(group)

# Using ThreadPoolExecutor for parallel processing with a limited number of threads
num_threads = 4  

with ThreadPoolExecutor(max_workers=num_threads) as executor:
    results = list(executor.map(parallel_process_group, grouped))

# Initializing an empty DataFrame for the results
frechet_df = pd.DataFrame(columns=['PID', 'AGVname', 'DRate', 'Point_ID', 'Frechet_Distance'])

# Collecting results from the parallel processing
for result in results:
    result_df = pd.DataFrame(result)
    frechet_df = pd.concat([frechet_df, result_df], ignore_index=True)

# Merging the frechet_df DataFrame with the original dataset using the keys
study_data = pd.merge(study_data, frechet_df, on=['PID', 'AGVname', 'DRate', 'Point_ID'], how='left')

# Printing summary of the processing
print(f"Finished processing {len(grouped)} groups.")
print(f"Total points processed: {len(study_data)}")

# Returning the updated study_data DataFrame with Fréchet Distance
study_data

In [ ]:
study_data.head()

In [ ]:
study_data.columns

## Visualizing the Fretchet Distance Calculation

### The main goal of this is to confirm that all generated trajectories are directed towards 'point 3' which is the final point for each interaction

In [ ]:
# Selecting a Sample Interaction Type
sample_pid = 10  # Example PID
sample_agv_number = 5  # Example AGV Number
sample_DRate = 'High'  # Example Behavior, enter either 'High' or 'Low' 

# Filtering the dataset for the chosen interaction
sample_interaction = study_data[(study_data['PID'] == sample_pid) &
                                (study_data['AGVname'] == sample_agv_number) &
                                (study_data['DRate'] == sample_DRate)]

# Checking if the filtered interaction is empty
if sample_interaction.empty:
    print(f"No data found for PID: {sample_pid}, AGVname: {sample_agv_number}, DRate: {sample_DRate}")
else:
    # Identifying Key Points within the Interaction
    first_point = sample_interaction.iloc[0]  # First actual coordinate
    middle_point = sample_interaction.iloc[len(sample_interaction) // 3]  # Middle actual coordinate, tune as you like
    close_last_point = sample_interaction.iloc[-3]  # Modify the variable to visualize a point backwards from the last point
    final_point = sample_interaction.iloc[-1]  # Final actual coordinate, this helps visualize whether the expected trajectories are actually directed to the final point

    key_points = [first_point, middle_point, close_last_point, final_point]  # Including the final point

    # Initializing the plot
    plt.figure(figsize=(12, 8))

    # Plotting the Trajectories and Key Points
    for i, point in enumerate(key_points):
        # Plotting the actual coordinate
        plt.scatter(point['User_X'], point['User_Y'], color='blue', marker='o', zorder=5, label='Actual Coordinate' if i == 0 else "")
        plt.annotate(f'Point {i+1}', (point['User_X'], point['User_Y']), textcoords="offset points", xytext=(0,10), ha='center')

        if i < (len(key_points) - 1):  # Skip generating expected trajectory for the final point
            # Generating and plotting expected trajectory points from the current actual coordinate
            expected_trajectory = generate_expected_trajectory((point['User_X'], point['User_Y']), sample_interaction.iloc[-1][['User_X', 'User_Y']], n_points)
            plt.plot(*zip(*expected_trajectory), 'r--', label='Expected Trajectory' if i == 0 else "")

            # Plotting the actual path points used for the Fréchet Distance calculation
            actual_path_points = select_actual_path_points((point['User_X'], point['User_Y']), sample_interaction[['User_X', 'User_Y']].values, n_points)
            plt.plot(*zip(*actual_path_points), 'g-', label='Actual Path Segment' if i == 0 else "")

        # Including Fréchet Distance annotations for all points
        plt.text(point['User_X'], point['User_Y'], f'Fréchet: {point["Frechet_Distance"]:.2f}', ha='right', va='bottom')

    print(middle_point) # Allows us to confirm whether the right coordinates are being pulled, compare User_X and Y to what is displayed on the plot

    # Formatting the plot
    plt.title(f'Analysis for PID: {sample_pid}, AGV Number: {sample_agv_number}, DRate: {sample_DRate}')
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.legend(loc='best')
    plt.grid(True)
    plt.show()

In [ ]:
study_data.to_csv("ISU_data_by_0.3_sec_for_R.csv", index = False)

## t-test between each pair of DVs

In [ ]:
group_t = study_data['Trust']
group_e = study_data['Expect']
group_s = study_data['Safe']
group_c = study_data['Comfort']

In [ ]:
t_stat, p_value = ttest_ind(group_t, group_e)

print(f"T-statistics between group_t and group_e: {t_stat:.2f}")
print(f"P-value between group_t and group_e: {p_value:.2f}")

In [ ]:
t_stat, p_value = ttest_ind(group_t, group_s)

print(f"T-statistics between group_t and group_s: {t_stat:.2f}")
print(f"P-value between group_t and group_s: {p_value:.2f}")

In [ ]:
t_stat, p_value = ttest_ind(group_t, group_c)

print(f"T-statistics between group_t and group_c: {t_stat:.2f}")
print(f"P-value between group_t and group_c: {p_value:.2f}")

In [ ]:
t_stat, p_value = ttest_ind(group_e, group_s)

print(f"T-statistics between group_e and group_s: {t_stat:.2f}")
print(f"P-value between group_e and group_s: {p_value:.2f}")

In [ ]:
t_stat, p_value = ttest_ind(group_e, group_c)

print(f"T-statistics between group_e and group_c: {t_stat:.2f}")
print(f"P-value between group_e and group_c: {p_value:.2f}")

In [ ]:
t_stat, p_value = ttest_ind(group_s, group_c)

print(f"T-statistics between group_s and group_c: {t_stat:.2f}")
print(f"P-value between group_s and group_c: {p_value:.2f}")

## Dependent Variables Correlation with User's Speed and Frechet Distance

In [ ]:
# Select the dependent variable to include in the correlation matrix
features = ['Comfort', #'Expect','Trust', 'Safe', 'Comfort',
           'Frechet_Distance', 'User_Relative_Speed']

# Create the correlation matrix
correlation_matrix = study_data[features].corr()

# Visualize the correlation matrix
plt.figure(figsize = (8, 6))
heatmap = sns.heatmap(correlation_matrix, cmap = 'coolwarm', annot=False, annot_kws = {'size': 20, 'color': 'b'}, fmt = '.2', vmin = -1, vmax = 1, linecolor = 'black', linewidth = '0.5')

# Adjust tick labels sizes
heatmap.tick_params(axis = 'x', labelsize = 14)
heatmap.tick_params(axis = 'y', labelsize = 14)
heatmap.set_xticklabels(heatmap.get_xticklabels(), rotation = 45, horizontalalignment = 'right')
heatmap.set_yticklabels(heatmap.get_yticklabels(), rotation = 0, horizontalalignment = 'right')
plt.title('Correlation Matrix for Dependent Variables')

# Define the filename with timestamp
filename = 'Correlation Matrix for Dependent Variables'
# plt.savefig(save_dir + filename, bbox_inches = 'tight', pad_inches=0.1)

plt.show()

correlation_matrix.round(3)

## Regression Models with Frechet Distance and User's Relative Speed as Features 

In [ ]:
X = study_data[['Frechet_Distance', 'User_Relative_Speed']]
y = study_data['Expect']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 2024)

print('Shape of X_train: ', X_train.shape)
print('Shape of X_test: ', X_test.shape)
print('Shape of y_train: ', y_train.shape)
print('Shape of y_test: ', y_test.shape)

In [ ]:
scaler = preprocessing.MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.fit_transform(X_test), columns=X_test.columns)

In [ ]:
Regressor_models = [ 
          ('Linear Regression', LinearRegression()),
          ('RegressionTree', DecisionTreeRegressor(max_depth=4, random_state= 2024)),
          ('KNN Regression', KNeighborsRegressor(n_neighbors=2)),
          ('RandomForestRegression', RandomForestRegressor(n_estimators = 20, random_state = 2024))
        ]

# Define lists to store results
Regressor_results = []
Regressor_names = []

# Define the header separately
header = ["Model", "Mean Absolute Error", "Mean Absolute Percent Error", "Mean Squared Error", "Root Mean Squared Error", "Relative RMSE", "R_Squared"]

# Train and evaluate each regression model
for name, model in Regressor_models:
    # Train the regression model
    fitted = model.fit(X_train_scaled, y_train)
    
    # Make predictions
    pred = fitted.predict(X_test_scaled)
    
    # Calculate Mean Absolute Percent Error (MAPE)
    current_mape = mape(y_test, pred)
    
    # Calculate Mean Squared Error (MSE)
    current_mse = mse(y_test, pred)
    
    # Calculate Root Mean Squared Error (RMSE)
    current_rmse = np.sqrt(current_mse)
    
    # Calculate Relative Root Mean Squared Error (RRMSE)
    rrmse_denominator = np.mean(y_test)
    current_rrmse = current_rmse / rrmse_denominator
    
    # Calculate Mean Absolute Error (MAE)
    current_mae = mae(y_test, pred)

    # Calculate R2-squared (r2)
    current_r2 = r2_score(y_test, pred)
    
    # Print the results
    print(tabulate([[name, f'{current_mae:.2f}', f'{current_mape:.2f}', f'{current_mse:.2f}', f'{current_rmse:.2f}', f'{current_rrmse:.2f}', f'{current_r2:.2f}']], header, tablefmt="fancy_grid"))
    
    # Append results to lists
    Regressor_results.append({
        'Model': name,
        'MAE': "{:.2f}".format(current_mae),
        'MAPE': "{:.2f}".format(current_mape),
        'MSE': "{:.2f}".format(current_mse),
        'RMSE': "{:.2f}".format(current_rmse),
        'RRMSE': "{:.2f}".format(current_rrmse),
        'R2-Squared': "{:.2f}".format(current_r2)
    })
    Regressor_names.append(name)

## Trend of Frechet Distance and User's Relative Speed across Users

In [ ]:
# Group Data by Point_ID
grouped_for_visualization = (study_data[['Point_ID', 'Frechet_Distance', 'User_Relative_Speed']].groupby(['Point_ID']).agg(
    FD_mean = ('Frechet_Distance', 'mean'),
    FD_std = ('Frechet_Distance', 'std'),
    FD_count = ('Frechet_Distance', 'count'),
    URS_mean = ('User_Relative_Speed', 'mean'),
    URS_std = ('User_Relative_Speed', 'std'),
    URS_count = ('User_Relative_Speed', 'count'),
    )).reset_index()

# grouped_for_visualization = grouped_for_visualization.droplevel(axis = 1, level =0).reset_index()

# Calculate the confidence intervals for frechet distance
grouped_for_visualization['FD_ci'] = 1.96*grouped_for_visualization['FD_std'] / np.sqrt(grouped_for_visualization['FD_count'])
grouped_for_visualization['FD_ci_lower'] = grouped_for_visualization['FD_mean'] - grouped_for_visualization['FD_ci']
grouped_for_visualization['FD_ci_upper'] = grouped_for_visualization['FD_mean'] + grouped_for_visualization['FD_ci']

# Calculate the confidence intervals for user's relative speed
grouped_for_visualization['URS_ci'] = 1.96*grouped_for_visualization['URS_std'] / np.sqrt(grouped_for_visualization['URS_count'])
grouped_for_visualization['URS_ci_lower'] = grouped_for_visualization['URS_mean'] - grouped_for_visualization['FD_ci']
grouped_for_visualization['URS_ci_upper'] = grouped_for_visualization['URS_mean'] + grouped_for_visualization['FD_ci']

grouped_for_visualization.head()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 4), sharex = True)

# Plot the mean line and confidence intervals for Frechet Distance
axs[0].plot(grouped_for_visualization['Point_ID'], grouped_for_visualization['FD_mean'], label = 'Average Frechet Distance', color = 'blue')
axs[0].fill_between(grouped_for_visualization['Point_ID'], grouped_for_visualization['FD_ci_lower'], grouped_for_visualization['FD_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Frechet Distance')
axs[0].set_ylabel('Frechet Distance (Unreal Unit, 2 CM)')
axs[0].set_xlabel('Point_ID')
axs[0].set_title('Frechect Distance by Point_ID')
axs[0].legend()
axs[0].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[0].set_ylim(0, grouped_for_visualization[['FD_mean','FD_ci_upper']].max().max() + 10)
axs[0].set_xlim(0, )

# Plot the mean line and confidence intervals for User's Relative Speed
axs[1].plot(grouped_for_visualization['Point_ID'], grouped_for_visualization['URS_mean'], label = 'Average Frechet Distance', color = 'orange')
axs[1].fill_between(grouped_for_visualization['Point_ID'], grouped_for_visualization['URS_ci_lower'], grouped_for_visualization['URS_ci_upper'], color='orange', alpha = 0.15, label = '95% CI for Frechet Distance')
axs[1].set_ylabel('User Relative Speed (Unreal Unit/S, 2*(CM/S)')
axs[1].set_xlabel('Point_ID')
axs[1].set_title('User Relative Speed by Point_ID')
axs[1].legend()
axs[1].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[1].set_ylim(0, grouped_for_visualization[['FD_mean','FD_ci_upper']].max().max() + 10)
axs[1].set_xlim(0, )

plt.legend()
# Add a light gray grid
plt.grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)

# Define the filename with timestamp
filename = 'Frechet Distance and User Relative Speed Trend over time'
# plt.savefig(save_dir + filename, bbox_inches = 'tight', pad_inches=0.1)

plt.xticks(rotation=45)
plt.show()

## Regression for Trust 

In [ ]:
# Get dummy variables for categorical features. If 'Cross_First', and 'Frechet_Distance' were filled out, don't drop them
study_data_with_dummies = pd.get_dummies(study_data.drop(columns=['User_X_Difference','User_Y_Difference','quantile']), columns=['PID','DRate'], drop_first=True)

In [ ]:
# Splitting the dataset into training (80%) and testing (20%) sets
train_data, test_data = train_test_split(study_data_with_dummies, test_size=0.2, random_state=42)

# Checking the shape of the training and testing sets
train_data.shape, test_data.shape

In [ ]:
# Separating the independent variables (X) and the target variable (y: Trust)
X_train = train_data.drop(columns=['Trust'])
y_train = train_data['Trust']

X_test = test_data.drop(columns=['Trust'])
y_test = test_data['Trust']

print('Shape of X_train: ', X_train.shape)
print('Shape of X_test: ', X_test.shape)
print('Shape of y_train: ', y_train.shape)
print('Shape of y_test: ', y_test.shape)

In [ ]:
# Preditors are scaled but the target (y) is not touched.
# We should scale both training and test partitions.
scaler = preprocessing.MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.fit_transform(X_test), columns=X_test.columns)

In [ ]:
Regressor_models = [ 
          ('Linear Regression', LinearRegression()),
          ('KNN Regression', KNeighborsRegressor(n_neighbors=2)),
          ('RegressionTree', DecisionTreeRegressor(max_depth=4, random_state= 2024)),
          ('RandomForestRegression', RandomForestRegressor(n_estimators = 20, random_state = 2024))
        ]

# Define lists to store results
Regressor_results = []
Regressor_names = []

# Define the header separately
header = ["Model", "Mean Absolute Error", "Mean Absolute Percent Error", "Mean Squared Error", "Root Mean Squared Error", "Relative RMSE"]

# Train and evaluate each regression model
for name, model in Regressor_models:
    # Train the regression model
    fitted = model.fit(X_train_scaled, y_train)
    
    # Make predictions
    pred = fitted.predict(X_test_scaled)
    
    # Calculate Mean Absolute Percent Error (MAPE)
    current_mape = mape(y_test, pred)
    
    # Calculate Mean Squared Error (MSE)
    current_mse = mse(y_test, pred)
    
    # Calculate Root Mean Squared Error (RMSE)
    current_rmse = np.sqrt(current_mse)
    
    # Calculate Relative Root Mean Squared Error (RRMSE)
    rrmse_denominator = np.mean(y_test)
    current_rrmse = current_rmse / rrmse_denominator
    
    # Calculate Mean Absolute Error (MAE)
    current_mae = mae(y_test, pred)
    
    # Print the results
    print(tabulate([[name, f'{current_mae:.2f}', f'{current_mape:.2f}', f'{current_mse:.2f}', f'{current_rmse:.2f}', f'{current_rrmse:.2f}']], header, tablefmt="fancy_grid"))
    
    # Append results to lists
    Regressor_results.append({
        'Model': name,
        'MAE': "{:.2f}".format(current_mae),
        'MAPE': "{:.2f}".format(current_mape),
        'MSE': "{:.2f}".format(current_mse),
        'RMSE': "{:.2f}".format(current_rmse),
        'RRMSE': "{:.2f}".format(current_rrmse)
    })
    Regressor_names.append(name)

### Feature Selection
#### The results clearly indicate the models are overfitting. Thus feature selection must be done.

In [ ]:
study_data.keys()

In [ ]:
# OneHotEncoder to convert string feature to numerical
one_hot_encoder = OneHotEncoder()
encoded_feature = one_hot_encoder.fit_transform(study_data[['DRate']]).toarray()
df_encoded = pd.DataFrame(encoded_feature, columns = one_hot_encoder.get_feature_names_out(['DRate']))
study_data_encoded = study_data.join(df_encoded).drop(columns=['DRate','Trust', 'PID','quantile'], axis=1)

In [ ]:
features = study_data_encoded

scaler = preprocessing.MinMaxScaler()
scaler.fit(features)

scaled_features = scaler.transform(features)
scaled_features_df = pd.DataFrame(scaled_features, columns = features.columns)

In [ ]:
scaled_features_df.head()

In [ ]:
# Initialize PCA with the same number of components as before
pca = PCA(n_components=9)
pca_result = pca.fit_transform(scaled_features_df)
pca_df = pd.DataFrame(data = pca_result, columns=['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9'])

# Get the explained variance ratio for each principal component
explained_variance_ratio = pca.explained_variance_ratio_

# Print the explained variance ratio for each principal component
for i, ratio in enumerate(explained_variance_ratio):
    print(f'Explained Variance Ratio for PC{i + 1}: {ratio:.2f}')

# Calculate the cumulative explained variance ratio
cumulative_variance_ratio = np.cumsum(explained_variance_ratio)

# Print the cumulative explained variance ratio
print('\nCumulative Explained Variance Ratio:')
for i, ratio in enumerate(cumulative_variance_ratio):
    print(f'PC{i + 1}: {ratio:.2f}')

# Determine which features to keep based on the cumulative explained variance ratio
num_components_to_keep = np.argmax(cumulative_variance_ratio >= 0.95) + 1
print(f'\nNumber of components to keep for 95% variance: {num_components_to_keep}')

# Determine which features contribute the most to each principal component
top_features_per_component = {}
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

for pc, loading in enumerate(loadings.T):
    top_features = scaled_features_df.columns[np.argsort(np.abs(loading))[::-1]][:3]  # Top 3 features per component
    top_features_per_component[f'PC{pc + 1}'] = top_features.tolist()

# Plot the explained variance and cumulative explained variance
plt.figure(figsize=(8, 6))
plt.plot(np.arange(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker='o', label='Explained Variance Ratio')
plt.plot(np.arange(1, len(cumulative_variance_ratio) + 1), cumulative_variance_ratio, marker='s', label='Cumulative Explained Variance Ratio')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% Variance Threshold')

# Annotate the plot with the top contributing features for each principal component
for pc, top_features in top_features_per_component.items():
    annotation_text = f'{pc}: {", ".join(top_features)}'
    plt.annotate(annotation_text, xy=(int(pc[2:]), 0.95), xytext=(int(pc[2:]), 0.7 - int(pc[2:]) * 0.06)
                 #,arrowprops=dict(facecolor='black', arrowstyle='<-')
                 ,fontsize=10, ha='center')

plt.xlabel('Number of Components')
plt.ylabel('Variance Ratio')
plt.title('Explained Variance and Cumulative Explained Variance')
plt.legend(loc='upper left', bbox_to_anchor=(1, 0.5))
plt.grid(True)
plt.tight_layout()

# Generate a timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Define the filename with timestamp
filename = f"Explained Variance and Cumulative Explained Variance_{timestamp}.png"

plt.savefig(save_dir + filename, dpi = 600)

plt.show()

### Running Regression Models with New PCs

In [ ]:
X_of_pca = pca_df[['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9']]
y_of_pca = study_data['Trust']

X_of_pca_train, X_of_pca_test, y_of_pca_train, y_of_pca_test = train_test_split(X_of_pca, y_of_pca, test_size=0.2, random_state=2024)

In [ ]:
Regressor_models = [ 
          ('Linear Regression', LinearRegression()),
          ('KNN Regression', KNeighborsRegressor(n_neighbors=2)),
          ('RegressionTree', DecisionTreeRegressor(max_depth=4, random_state= 2024)),
          ('RandomForestRegression', RandomForestRegressor(n_estimators = 20, random_state = 2024))
        ]

# Define lists to store results
Regressor_results = []
Regressor_names = []

# Define the header separately
header = ["Model", "Mean Absolute Error", "Mean Absolute Percent Error", "Mean Squared Error", "Root Mean Squared Error", "Relative RMSE"]

# Train and evaluate each regression model
for name, model in Regressor_models:
    # Train the regression model
    fitted = model.fit(X_of_pca_train, y_of_pca_train)
    
    # Make predictions
    pred = fitted.predict(X_of_pca_test)

    # Calculate Mean Absolute Error (MAE)
    current_mae = mae(y_of_pca_test, pred)
    
    # Calculate Mean Absolute Percent Error (MAPE)
    current_mape = mape(y_of_pca_test, pred)
    
    # Calculate Mean Squared Error (MSE)
    current_mse = mse(y_of_pca_test, pred)
    
    # Calculate Root Mean Squared Error (RMSE)
    current_rmse = np.sqrt(current_mse)
    
    # Calculate Relative Root Mean Squared Error (RRMSE)
    rrmse_denominator = np.mean(y_of_pca_test)
    current_rrmse = current_rmse / rrmse_denominator
    
    # Print the results
    print(tabulate([[name, f'{current_mae:.2f}', f'{current_mape:.2f}', f'{current_mse:.2f}', f'{current_rmse:.2f}', f'{current_rrmse:.2f}']], header, tablefmt="fancy_grid"))
    
    # Append results to lists
    Regressor_results.append({
        'Model': name,
        'MAE': "{:.2f}".format(current_mae),
        'MAPE': "{:.2f}".format(current_mape),
        'MSE': "{:.2f}".format(current_mse),
        'RMSE': "{:.2f}".format(current_rmse),
        'RRMSE': "{:.2f}".format(current_rrmse)
    })
    Regressor_names.append(name)

## Interpolating the Trust Value based on Features

In [ ]:
# Sort the dataset
study_data_sorted = study_data.sort_values(by=['PID', 'AGVname', 'DRate', 'Point_ID'], ignore_index=True)
study_data_sorted.loc[:,['PID', 'AGVname', 'DRate', 'Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head()
# study_data_copied = study_data_sorted.copy()
# study_data_copied['Trust'] = np.nan

In [ ]:
# Group the data
grouped_study_data_sorted = study_data_sorted.groupby(['PID', 'AGVname', 'DRate'])
study_data_sorted.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head()

In [ ]:
# calculating the length of the series 
#len = grouped_study_data_sorted['Trust'].size 
# Loop over the groups
for (pid, agvname, drate), group in grouped_study_data_sorted:
    indices = group.index.tolist()
    #trust_values = group['Trust'].values
    for idx in indices[1:-1]:
         study_data_sorted.at[idx, 'Trust'] = np.nan
        
    #first_index = study_data_sorted.at[indices[0], 'Trust']
    #print(first_index)
    #last_index = study_data_sorted.at[indices[-1], 'Trust']
    #print(last_index)

    #study_data_sorted['Trust'] = np.nan
    
    #study_data_sorted.at[indices[0], 'Trust']  = first_index
    #study_data_sorted.at[indices[-1], 'Trust'] = last_index
study_data_sorted.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head(180)

### Linear Interpolation

In [ ]:
study_data_linearly_interpolated = study_data_sorted.interpolate(method='linear', limit_direction='forward', axis=0)
study_data_linearly_interpolated.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head(180)

### SLinear Interpolation

In [ ]:
study_data_slinearly_interpolated = study_data_sorted.interpolate(method='slinear', limit_direction='forward', axis=0)
study_data_slinearly_interpolated.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head(180)

### Quadratic Interpolation

In [ ]:
study_data_quadratically_interpolated = study_data_sorted.interpolate(method='quadratic', order=2)
study_data_quadratically_interpolated.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head(180)

### Cubic Interpolation

In [ ]:
study_data_cubicly_interpolated = study_data_sorted.interpolate(method='cubic', order=3)
study_data_cubicly_interpolated.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head(180)

### Polynomial Interpolation

In [ ]:
study_data_polynomial_interpolated = study_data_sorted.interpolate(method='polynomial', order=5)
study_data_polynomial_interpolated.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head(180)

### Visualizing the Interpolation

In [ ]:
# Group Data by Point_ID
grouped_for_study_data_linearly_interpolated = (study_data_linearly_interpolated[['Point_ID', 'Trust']].groupby(['Point_ID']).agg(
    Trust_mean = ('Trust', 'mean'),
    Trust_std = ('Trust', 'std'),
    Trust_count = ('Trust', 'count')
    )).reset_index()

# grouped_for_visualization = grouped_for_visualization.droplevel(axis = 1, level =0).reset_index()

# Calculate the confidence intervals for trust
grouped_for_study_data_linearly_interpolated['Trust_ci'] = 1.96*grouped_for_study_data_linearly_interpolated['Trust_std'] / np.sqrt(grouped_for_study_data_linearly_interpolated['Trust_count'])
grouped_for_study_data_linearly_interpolated['Trust_ci_lower'] = grouped_for_study_data_linearly_interpolated['Trust_mean'] - grouped_for_study_data_linearly_interpolated['Trust_ci']
grouped_for_study_data_linearly_interpolated['Trust_ci_upper'] = grouped_for_study_data_linearly_interpolated['Trust_mean'] + grouped_for_study_data_linearly_interpolated['Trust_ci']

# Group Data by Point_ID
grouped_for_study_data_slinearly_interpolated = (study_data_slinearly_interpolated[['Point_ID', 'Trust']].groupby(['Point_ID']).agg(
    Trust_mean = ('Trust', 'mean'),
    Trust_std = ('Trust', 'std'),
    Trust_count = ('Trust', 'count')
    )).reset_index()

# grouped_for_visualization = grouped_for_visualization.droplevel(axis = 1, level =0).reset_index()

# Calculate the confidence intervals for trust
grouped_for_study_data_slinearly_interpolated['Trust_ci'] = 1.96*grouped_for_study_data_slinearly_interpolated['Trust_std'] / np.sqrt(grouped_for_study_data_slinearly_interpolated['Trust_count'])
grouped_for_study_data_slinearly_interpolated['Trust_ci_lower'] = grouped_for_study_data_slinearly_interpolated['Trust_mean'] - grouped_for_study_data_slinearly_interpolated['Trust_ci']
grouped_for_study_data_slinearly_interpolated['Trust_ci_upper'] = grouped_for_study_data_slinearly_interpolated['Trust_mean'] + grouped_for_study_data_slinearly_interpolated['Trust_ci']

# Group Data by Point_ID
grouped_for_study_data_quadratically_interpolated = (study_data_quadratically_interpolated[['Point_ID', 'Trust']].groupby(['Point_ID']).agg(
    Trust_mean = ('Trust', 'mean'),
    Trust_std = ('Trust', 'std'),
    Trust_count = ('Trust', 'count')
    )).reset_index()

# grouped_for_visualization = grouped_for_visualization.droplevel(axis = 1, level =0).reset_index()

# Calculate the confidence intervals for trust
grouped_for_study_data_quadratically_interpolated['Trust_ci'] = 1.96*grouped_for_study_data_quadratically_interpolated['Trust_std'] / np.sqrt(grouped_for_study_data_quadratically_interpolated['Trust_count'])
grouped_for_study_data_quadratically_interpolated['Trust_ci_lower'] = grouped_for_study_data_quadratically_interpolated['Trust_mean'] - grouped_for_study_data_quadratically_interpolated['Trust_ci']
grouped_for_study_data_quadratically_interpolated['Trust_ci_upper'] = grouped_for_study_data_quadratically_interpolated['Trust_mean'] + grouped_for_study_data_quadratically_interpolated['Trust_ci']

# Group Data by Point_ID
grouped_for_study_data_cubicly_interpolated = (study_data_cubicly_interpolated[['Point_ID', 'Trust']].groupby(['Point_ID']).agg(
    Trust_mean = ('Trust', 'mean'),
    Trust_std = ('Trust', 'std'),
    Trust_count = ('Trust', 'count')
    )).reset_index()

# grouped_for_visualization = grouped_for_visualization.droplevel(axis = 1, level =0).reset_index()

# Calculate the confidence intervals for trust
grouped_for_study_data_cubicly_interpolated['Trust_ci'] = 1.96*grouped_for_study_data_cubicly_interpolated['Trust_std'] / np.sqrt(grouped_for_study_data_cubicly_interpolated['Trust_count'])
grouped_for_study_data_cubicly_interpolated['Trust_ci_lower'] = grouped_for_study_data_cubicly_interpolated['Trust_mean'] - grouped_for_study_data_cubicly_interpolated['Trust_ci']
grouped_for_study_data_cubicly_interpolated['Trust_ci_upper'] = grouped_for_study_data_cubicly_interpolated['Trust_mean'] + grouped_for_study_data_cubicly_interpolated['Trust_ci']

# Group Data by Point_ID
grouped_for_study_data_polynomial_interpolated = (study_data_polynomial_interpolated[['Point_ID', 'Trust']].groupby(['Point_ID']).agg(
    Trust_mean = ('Trust', 'mean'),
    Trust_std = ('Trust', 'std'),
    Trust_count = ('Trust', 'count')
    )).reset_index()

# grouped_for_visualization = grouped_for_visualization.droplevel(axis = 1, level =0).reset_index()

# Calculate the confidence intervals for trust
grouped_for_study_data_polynomial_interpolated['Trust_ci'] = 1.96*grouped_for_study_data_polynomial_interpolated['Trust_std'] / np.sqrt(grouped_for_study_data_polynomial_interpolated['Trust_count'])
grouped_for_study_data_polynomial_interpolated['Trust_ci_lower'] = grouped_for_study_data_polynomial_interpolated['Trust_mean'] - grouped_for_study_data_polynomial_interpolated['Trust_ci']
grouped_for_study_data_polynomial_interpolated['Trust_ci_upper'] = grouped_for_study_data_polynomial_interpolated['Trust_mean'] + grouped_for_study_data_polynomial_interpolated['Trust_ci']

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(32, 4), sharex = True)

# Plot the mean line and confidence intervals for linearly interpolated trust
axs[0].plot(grouped_for_study_data_linearly_interpolated['Point_ID'], grouped_for_study_data_linearly_interpolated['Trust_mean'], label = 'Average Trust', color = 'orange')
axs[0].fill_between(grouped_for_study_data_linearly_interpolated['Point_ID'], grouped_for_study_data_linearly_interpolated['Trust_ci_lower'], grouped_for_study_data_linearly_interpolated['Trust_ci_upper'], color='orange', alpha = 0.15, label = '95% CI for Trust')
axs[0].set_ylabel('Trust')
axs[0].set_xlabel('Point_ID')
axs[0].set_title('Linearly Interpolated Trust by Point_ID')
axs[0].legend()
axs[0].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[0].set_ylim(grouped_for_study_data_cubicly_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_linearly_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[0].set_xlim(0, )

# Plot the mean line and confidence intervals for slinearly interpolated trust
axs[1].plot(grouped_for_study_data_slinearly_interpolated['Point_ID'], grouped_for_study_data_slinearly_interpolated['Trust_mean'], label = 'Average Trust', color = 'orange')
axs[1].fill_between(grouped_for_study_data_slinearly_interpolated['Point_ID'], grouped_for_study_data_slinearly_interpolated['Trust_ci_lower'], grouped_for_study_data_slinearly_interpolated['Trust_ci_upper'], color='orange', alpha = 0.15, label = '95% CI for Trust')
axs[1].set_ylabel('Trust')
axs[1].set_xlabel('Point_ID')
axs[1].set_title('SLinearly Interpolated Trust by Point_ID')
axs[1].legend()
axs[1].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[1].set_ylim(grouped_for_study_data_cubicly_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_slinearly_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[1].set_xlim(0, )

# Plot the mean line and confidence intervals for quadratically interpolated trust
axs[2].plot(grouped_for_study_data_quadratically_interpolated['Point_ID'], grouped_for_study_data_quadratically_interpolated['Trust_mean'], label = 'Average Trust', color = 'blue')
axs[2].fill_between(grouped_for_study_data_quadratically_interpolated['Point_ID'], grouped_for_study_data_quadratically_interpolated['Trust_ci_lower'], grouped_for_study_data_quadratically_interpolated['Trust_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Trust')
axs[2].set_ylabel('Trust')
axs[2].set_xlabel('Point_ID')
axs[2].set_title('Quadratically (O=2) Interpolated Trust by Point_ID')
axs[2].legend()
axs[2].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[2].set_ylim(grouped_for_study_data_quadratically_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_quadratically_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[2].set_xlim(0, )

# Plot the mean line and confidence intervals for cubicly interpolated trust
axs[3].plot(grouped_for_study_data_cubicly_interpolated['Point_ID'], grouped_for_study_data_cubicly_interpolated['Trust_mean'], label = 'Average Trust', color = 'blue')
axs[3].fill_between(grouped_for_study_data_cubicly_interpolated['Point_ID'], grouped_for_study_data_cubicly_interpolated['Trust_ci_lower'], grouped_for_study_data_cubicly_interpolated['Trust_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Trust')
axs[3].set_ylabel('Trust')
axs[3].set_xlabel('Point_ID')
axs[3].set_title('Cubicly (O=3) Interpolated Trust by Point_ID')
axs[3].legend()
axs[3].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[3].set_ylim(grouped_for_study_data_cubicly_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_cubicly_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[3].set_xlim(0, )

# Plot the mean line and confidence intervals for polynomial interpolated trust
axs[4].plot(grouped_for_study_data_polynomial_interpolated['Point_ID'], grouped_for_study_data_polynomial_interpolated['Trust_mean'], label = 'Average Trust', color = 'blue')
axs[4].fill_between(grouped_for_study_data_polynomial_interpolated['Point_ID'], grouped_for_study_data_polynomial_interpolated['Trust_ci_lower'], grouped_for_study_data_polynomial_interpolated['Trust_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Trust')
axs[4].set_ylabel('Trust')
axs[4].set_xlabel('Point_ID')
axs[4].set_title('Polynomial (O=5) Interpolated Trust by Point_ID')
axs[4].legend()
axs[4].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[4].set_ylim(grouped_for_study_data_polynomial_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_polynomial_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[4].set_xlim(0, )

plt.legend()
# Add a light gray grid
plt.grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)

# Define the filename with timestamp
filename = 'Interpolated Trust over Time with the same ylim'
plt.savefig(save_dir + filename, bbox_inches = 'tight', pad_inches=0.1)

plt.xticks(rotation=45)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(32, 4), sharex = True)

# Plot the mean line and confidence intervals for linearly interpolated trust
axs[0].plot(grouped_for_study_data_linearly_interpolated['Point_ID'], grouped_for_study_data_linearly_interpolated['Trust_mean'], label = 'Average Trust', color = 'orange')
axs[0].fill_between(grouped_for_study_data_linearly_interpolated['Point_ID'], grouped_for_study_data_linearly_interpolated['Trust_ci_lower'], grouped_for_study_data_linearly_interpolated['Trust_ci_upper'], color='orange', alpha = 0.15, label = '95% CI for Trust')
axs[0].set_ylabel('Trust')
axs[0].set_xlabel('Point_ID')
axs[0].set_title('Linearly Interpolated Trust by Point_ID')
axs[0].legend()
axs[0].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[0].set_ylim(grouped_for_study_data_linearly_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_linearly_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[0].set_xlim(0, )

# Plot the mean line and confidence intervals for slinearly interpolated trust
axs[1].plot(grouped_for_study_data_slinearly_interpolated['Point_ID'], grouped_for_study_data_slinearly_interpolated['Trust_mean'], label = 'Average Trust', color = 'orange')
axs[1].fill_between(grouped_for_study_data_slinearly_interpolated['Point_ID'], grouped_for_study_data_slinearly_interpolated['Trust_ci_lower'], grouped_for_study_data_slinearly_interpolated['Trust_ci_upper'], color='orange', alpha = 0.15, label = '95% CI for Trust')
axs[1].set_ylabel('Trust')
axs[1].set_xlabel('Point_ID')
axs[1].set_title('SLinearly Interpolated Trust by Point_ID')
axs[1].legend()
axs[1].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[1].set_ylim(grouped_for_study_data_slinearly_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_slinearly_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[1].set_xlim(0, )

# Plot the mean line and confidence intervals for quadratically interpolated trust
axs[2].plot(grouped_for_study_data_quadratically_interpolated['Point_ID'], grouped_for_study_data_quadratically_interpolated['Trust_mean'], label = 'Average Trust', color = 'blue')
axs[2].fill_between(grouped_for_study_data_quadratically_interpolated['Point_ID'], grouped_for_study_data_quadratically_interpolated['Trust_ci_lower'], grouped_for_study_data_quadratically_interpolated['Trust_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Trust')
axs[2].set_ylabel('Trust')
axs[2].set_xlabel('Point_ID')
axs[2].set_title('Quadratically (O=2) Interpolated Trust by Point_ID')
axs[2].legend()
axs[2].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[2].set_ylim(grouped_for_study_data_quadratically_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_quadratically_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[2].set_xlim(0, )

# Plot the mean line and confidence intervals for cubicly interpolated trust
axs[3].plot(grouped_for_study_data_cubicly_interpolated['Point_ID'], grouped_for_study_data_cubicly_interpolated['Trust_mean'], label = 'Average Trust', color = 'blue')
axs[3].fill_between(grouped_for_study_data_cubicly_interpolated['Point_ID'], grouped_for_study_data_cubicly_interpolated['Trust_ci_lower'], grouped_for_study_data_cubicly_interpolated['Trust_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Trust')
axs[3].set_ylabel('Trust')
axs[3].set_xlabel('Point_ID')
axs[3].set_title('Cubicly (O=3) Interpolated Trust by Point_ID')
axs[3].legend()
axs[3].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[3].set_ylim(grouped_for_study_data_cubicly_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_cubicly_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[3].set_xlim(0, )

# Plot the mean line and confidence intervals for polynomial interpolated trust
axs[4].plot(grouped_for_study_data_polynomial_interpolated['Point_ID'], grouped_for_study_data_polynomial_interpolated['Trust_mean'], label = 'Average Trust', color = 'blue')
axs[4].fill_between(grouped_for_study_data_polynomial_interpolated['Point_ID'], grouped_for_study_data_polynomial_interpolated['Trust_ci_lower'], grouped_for_study_data_polynomial_interpolated['Trust_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Trust')
axs[4].set_ylabel('Trust')
axs[4].set_xlabel('Point_ID')
axs[4].set_title('Polynomial (O=5) Interpolated Trust by Point_ID')
axs[4].legend()
axs[4].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[4].set_ylim(grouped_for_study_data_polynomial_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_polynomial_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[4].set_xlim(0, )

plt.legend()
# Add a light gray grid
plt.grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)

# Define the filename with timestamp
filename = 'Interpolated Trust over Time with the different ylim'
plt.savefig(save_dir + filename, bbox_inches = 'tight', pad_inches=0.1)

plt.xticks(rotation=45)
plt.show()